# LAB-HW-07 — BRAM Neuron State

**One new thing today:** turn the abstract `state[address]` from Lesson 9 into real on-chip Block RAM (BRAM) on KV260.

Prerequisites: LSN-009, LAB-HW-06, and a working LAB-HW-06 PS/Linux runtime path.

**Project Trace:** RMD-013 · T-HW-007/T-HW-011

## 1. Freeze one small state-memory contract

For this Lab only:

| item | frozen value |
|---|---:|
| state word | 32 bits |
| word count | 1024 |
| logical capacity | 4096 bytes / 4 KiB |
| valid word index | 0..1023 |
| PS-visible base | `0xA0000000` |
| word `i` byte offset | `4 * i` |

A 32-bit word is enough to teach addressable neuron state without deciding the final formal neuron record.

This Lab does **not** declare MOD-004 complete. LAB-HW-08 network replay and external DDR are both out of scope; this chapter is only about real on-chip BRAM state.

## 2. Why BRAM instead of a big pile of registers?

A register array can hold state, but FPGA devices also contain dedicated memory blocks. Vivado can infer those blocks from suitable synchronous RAM RTL.

For the teaching store:

- writes happen on a clock edge;
- reads are synchronous;
- `(* ram_style = "block" *)` states the implementation intent;
- the build must still prove the result by finding a RAMB18/RAMB36 primitive.

The attribute is an intent, **not** evidence by itself.

## 3. The real KV260 path

<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="250" viewBox="0 0 1000 250" role="img" aria-label="KV260 LAB-HW-07 PS Linux to BRAM state path">
  <rect x="20" y="75" width="150" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="95" y="108" text-anchor="middle" font-size="14">Ubuntu / Python</text>
  <text x="95" y="132" text-anchor="middle" font-size="12">fixed /dev/mem</text>
  <rect x="205" y="75" width="155" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="282" y="108" text-anchor="middle" font-size="14">PS HPM0 FPD</text>
  <text x="282" y="132" text-anchor="middle" font-size="12">AXI master</text>
  <rect x="395" y="75" width="145" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="468" y="118" text-anchor="middle" font-size="14">SmartConnect</text>
  <rect x="575" y="75" width="175" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="662" y="106" text-anchor="middle" font-size="14">AXI BRAM Controller</text>
  <text x="662" y="132" text-anchor="middle" font-size="12">0xA0000000 / 4 KiB</text>
  <rect x="785" y="75" width="195" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="882" y="105" text-anchor="middle" font-size="14">neuron state BRAM</text>
  <text x="882" y="130" text-anchor="middle" font-size="12">1024 × 32-bit</text>
  <path d="M170 117 L205 117 M360 117 L395 117 M540 117 L575 117 M750 117 L785 117" stroke="#333" stroke-width="2"/>
  <polygon points="205,117 195,112 195,122" fill="#333"/><polygon points="395,117 385,112 385,122" fill="#333"/>
  <polygon points="575,117 565,112 565,122" fill="#333"/><polygon points="785,117 775,112 775,122" fill="#333"/>
</svg>

AMD/Xilinx's K26 `base_gpio_bram` reference also places its BRAM window at `0xA0000000`.

AXI BRAM Controller is a platform adapter. You do not implement AXI in this Lab.

## 4. The important timing idea: synchronous read

The native memory port is clocked.

If address 7 is selected between clock edges, the output does not become the state at address 7 merely because the address wire changed. The memory operation happens on the active clock edge.

The teaching RTL also uses **read-first** behavior: if one clock edge both reads and writes the same address, the registered read output receives the old value, while the memory stores the new value.

Do not confuse this with Python MMIO timing. The AXI/controller/software stack hides several cycles. The Python test proves functional memory correctness, not “one-cycle host latency.”

## 5. Prove native memory behavior before touching the board

Run the open-source test:

```bash
iverilog -g2012 \
  -s kv260_neuron_state_store_tb \
  -o /tmp/lab07_state \
  boards/kv260/rtl/kv260_neuron_state_store.sv \
  boards/kv260/tb/kv260_neuron_state_store_tb.sv

vvp /tmp/lab07_state
```

Expected final line:

`PASS: kv260_neuron_state_store synchronous multi-address BRAM semantics`

The test covers multiple addresses, synchronous read, read-first behavior, and neighbor preservation.

## 6. Build the dedicated LAB-HW-07 bitstream

On the development host:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-07-build.log \
  -source boards/kv260/scripts/build_lab07_bram_state.tcl
```

The build must not stop at “synthesis succeeded.” Before writing the bitstream it requires:

- no DRC errors;
- non-negative setup slack;
- non-negative hold slack;
- `BRAM_PRIMITIVE_COUNT >= 1`.

Keep `utilization.rpt`. The resource report is part of the Lab oracle.

## 7. Program PL without power-cycling Linux

LAB-HW-06 already established the two-host boundary.

On the runtime host, unload an active Kria app if needed:

```bash
sudo xmutil unloadapp
```

Then on the development host:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-07-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-07/kv260_bram_state.bit
```

Do not power-cycle afterward; keep PS/Linux running while replacing only PL configuration.

## 8. Check the host oracle without hardware

```bash
python boards/kv260/runtime/state_bram_mmio.py \
  --dry-run \
  --json-out /tmp/lab-hw-07-dry-run.json
```

A dry-run must end in `STATUS=PASS`.

It checks the host-side address/value oracle only. It proves neither BRAM inference nor a real KV260 memory transaction.

## 9. Run the real multi-address state check

Copy `state_bram_mmio.py` to PS/Linux by an already-working method. Then:

```bash
sudo python3 /tmp/state_bram_mmio.py \
  --json-out /tmp/lab-hw-07-trace.json
```

The checker writes distinct values to indices:

`0, 1, 7, 31, 255, 511, 1023`

Then it:

1. reads every written location;
2. rewrites selected locations;
3. reads all tracked locations again;
4. proves untouched locations preserved their state.

Physical success requires final `STATUS=PASS`.

## 10. Failure classes

The host checker distinguishes:

- `TRANSPORT_DEVICE_MISSING`;
- `TRANSPORT_REQUIRES_ROOT`;
- `TRANSPORT_PERMISSION_OR_POLICY`;
- `TRANSPORT_MMAP_FAILED`;
- `STATE_READBACK_OR_ALIAS_MISMATCH`.

The Vivado build separately distinguishes `NO_BLOCK_RAM_PRIMITIVE` from DRC/timing/build failures.

If Ubuntu policy blocks `/dev/mem`, do not weaken system security. Preserve the evidence and keep T-HW-007 blocked.

If `BRAM_PRIMITIVE_COUNT=0`, do not call a working LUT memory “BRAM enough.” Fix the implementation/resource mapping.

## 11. Expected Evidence / Save Evidence

Retain:

- `lab-hw-07-build.log`;
- `timing_summary.rpt`, `utilization.rpt`, and `drc.rpt`;
- reported `RAMB18_COUNT`, `RAMB36_COUNT`, and `BRAM_PRIMITIVE_COUNT`;
- bitstream SHA-256;
- `lab-hw-07-program.log`;
- base `0xA0000000`, 4 KiB window, 1024 × 32-bit geometry;
- SHA-256 of `state_bram_mmio.py`;
- complete runtime stdout;
- `lab-hw-07-trace.json`;
- Ubuntu/kernel identity, board/carrier revision, Git commit/date.

A simulation PASS, dry-run PASS, or resource report alone never substitutes for the physical T-HW-007 roundtrip evidence.

## 12. Human Check

Explain:

1. Why is `state[address]` a memory problem rather than a neuron-arithmetic problem?
2. Why does changing a native BRAM address not imply asynchronous read data?
3. What does read-first mean for a same-cycle read/write?
4. Why does `ram_style="block"` not by itself prove BRAM was used?
5. Why are `0xA0000000 + 4*i` and word index `i` different quantities?
6. Why can Python/MMIO prove addressable memory correctness but not native one-cycle latency?
7. Why does LAB-HW-07 not mean formal MOD-004 is complete?

## 13. Official basis

- AMD Vivado Design Suite User Guide: Synthesis (UG901), 2026.1 — dedicated Block RAM uses synchronous reads; Vivado supports RAM inference and `RAM_STYLE`.
- AMD AXI Block RAM Controller Product Guide (PG078) — AXI4-Lite mode provides 32-bit single-beat access to BRAM and supports single-port BRAM configuration.
- AMD/Xilinx `kria-base-hardware`, K26 `base_gpio_bram/scripts/config_bd.tcl` — PS `M_AXI_HPM0_FPD` → SmartConnect → AXI BRAM Controller reference path and BRAM address `0xA0000000`.

The repository freezes this as a teaching path. Real T-HW-007 PASS still requires a physical KV260 run.